# Week 1 results: strategies vs benchmarks on the default universe

Everything below is computed by the tested functions in `src/gpl/`. This notebook only wires
them together and draws charts. **Not investment advice.**

- **Universe:** GCC equity (KSA, UAE, QAT), global equity (SPY, EFA, EEM), bonds (AGG, TLT), gold (GLD).
  US-listed ETFs priced in USD.
- **Strategies:** equal weight, minimum variance, maximum Sharpe, risk parity. Each is re-estimated
  at every month end on the trailing 252 trading days, applied from the next day, with 10 bps
  costs on turnover and a 40% cap per asset.
- **Benchmarks:** 60/40 (SPY/AGG) and SPY, rebalanced monthly through the same engine.

In [ ]:
import logging

import pandas as pd

from gpl import backtest as bt
from gpl.data import clean_prices, get_prices, get_risk_free, load_universe, to_returns
from gpl.metrics import drawdown_series, summary_table
from gpl.montecarlo import goal_probability, simulate_paths, terminal_stats
from gpl.optimise import (
    efficient_frontier,
    max_sharpe,
    min_variance,
    portfolio_performance,
    risk_parity,
)
from gpl.plots import drawdown_chart, equity_chart, fan_chart, frontier_chart, weights_chart

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
pd.options.display.float_format = "{:,.4f}".format

cfg = load_universe()
d = cfg["defaults"]
ASSETS = cfg["assets"]

## 1. Data

In [ ]:
prices = clean_prices(get_prices(ASSETS, start=d["start"], end=d["end"]), max_ffill=d["max_ffill"])
returns = to_returns(prices)
rf = get_risk_free(start=d["start"], end=d["end"])
start, end = prices.index[0].date(), prices.index[-1].date()
print(f"Sample: {start} to {end} ({len(returns)} daily returns)")

The common sample starts when the youngest ETF (KSA, launched September 2015) has a price.
**Every number below depends on this sample period.**

## 2. Asset-level risk and return

In [ ]:
summary_table(returns, rf=rf, level=d["var_level"]).sort_values("sharpe", ascending=False)

## 3. Backtests

In [ ]:
strategies = {
    "Equal weight": bt.equal_weight,
    "Min variance": bt.min_variance_strategy(max_weight=d["max_weight"]),
    "Max Sharpe": bt.max_sharpe_strategy(rf=rf, max_weight=d["max_weight"]),
    "Risk parity": bt.risk_parity_strategy(),
    "60/40": bt.fixed_weights(cfg["benchmarks"]["sixty_forty"]),
    "SPY": bt.fixed_weights(cfg["benchmarks"]["spy"]),
}
results = {
    name: bt.run_backtest(
        prices, strategy, rebalance=d["rebalance"], lookback=d["lookback"], cost_bps=d["cost_bps"]
    )
    for name, strategy in strategies.items()
}
strat_returns = pd.DataFrame({name: res.returns for name, res in results.items()})
equity = pd.DataFrame({name: res.equity for name, res in results.items()})

# Average monthly turnover x 12, excluding the initial purchase from cash.
annual_turnover = pd.Series(
    {name: res.turnover.iloc[1:].mean() * 12 for name, res in results.items()},
    name="annual_turnover",
)
strategy_table = summary_table(strat_returns, rf=rf, level=d["var_level"]).join(annual_turnover)
strategy_table.sort_values("sharpe", ascending=False)

In [ ]:
equity_chart(equity, title="Growth of $1 (net of 10 bps costs)").show()
drawdown_chart(drawdown_series(strat_returns)).show()

In [ ]:
for name in ["Min variance", "Max Sharpe", "Risk parity"]:
    title = f"{name}: weights by asset group"
    weights_chart(results[name].weights, groups=cfg["groups"], title=title).show()

## 4. Efficient frontier (full sample, **in hindsight**)

This uses the whole sample's means and covariances, so it is a picture of what *would* have been
efficient, not something an investor could have known at the time. The backtests above are the
honest out-of-sample test.

In [ ]:
mu = returns.mean() * 252
cov = returns.cov() * 252
rf_avg = float(
    (1 + rf.loc[returns.index[0] :]).prod() ** (252 / len(rf.loc[returns.index[0] :])) - 1
)

frontier = efficient_frontier(mu, cov, n_points=40, max_weight=d["max_weight"])
portfolios = {
    "Min variance": min_variance(cov, d["max_weight"]),
    "Max Sharpe": max_sharpe(mu, cov, rf_avg, d["max_weight"]),
    "Risk parity": risk_parity(cov),
    "Equal weight": pd.Series(1 / len(ASSETS), index=ASSETS),
}
points = {}
for name, w in portfolios.items():
    ret, vol = portfolio_performance(w, mu, cov)
    points[name] = (vol, ret)
frontier_chart(frontier, points).show()
pd.DataFrame(portfolios).T

## 5. Monte Carlo: 10 years ahead with today's risk-parity weights

Bootstrap resamples whole historical days, so it keeps fat tails and cross-asset correlation.
GBM assumes normal log returns. Both hold the weights fixed, rebalanced daily.

In [ ]:
w_now = results["Risk parity"].weights.iloc[-1]
horizon = 252 * 10
mc = {}
for method in ["bootstrap", "gbm"]:
    paths = simulate_paths(returns, w_now, horizon, n_paths=5_000, method=method, seed=42)
    stats = terminal_stats(paths)
    stats["P(>= 1.5x)"] = goal_probability(paths, 1.5)
    stats["P(< 1.0x)"] = 1 - goal_probability(paths, 1.0)
    mc[method] = stats
    if method == "bootstrap":
        fan_chart(paths, title="Risk parity: simulated growth of $1 over 10 years").show()
pd.DataFrame(mc)

## Limitations

- **Short sample:** about 9-10 years, dominated by one long US bull market, COVID and the 2022 bond
  sell-off. Results may not generalise.
- **Max Sharpe** relies on trailing mean returns, which are noisy estimates; expect it to chase recent
  winners and trade more.
- **Survivorship:** the universe is a fixed list of ETFs chosen today.
- **Costs:** 10 bps is a simple assumption. It ignores bid-ask spreads on thinner GCC ETFs, taxes and
  fund fees beyond the ETF's own expense ratio (already in the adjusted prices).
- **Monte Carlo** assumes the future looks like a reshuffle (bootstrap) or a normal version (GBM) of
  this sample.